In [3]:
import numpy as np
from ase.io import read, write
import matplotlib.pyplot as plt
from weas_widget import WeasWidget 

# Generate a P4 dimer where one of the molecules is rotated by 180 degrees by running
# `python generate_p4_dimer.py`
# in the terminal.

In [16]:
# Visualize the dimer
atoms = read("p4_dimer.xyz",index=":")
viewer = WeasWidget()
viewer.avr.model_style = 1 # ball & stick mode
viewer.from_ase(atoms)
viewer.avr.show_bonded_atoms = True # show bonds across periodic boundaries
viewer

WeasWidget(children=(<weas_widget.base_widget.BaseWidget object at 0x7f17e33f6590>,))

# Run TurboGAP with Tkatchenko-Scheffler (TS) corrections for the different rotations of the dimer by using `input_dimer_ts` as input:
# `turbogap_vdw predict > out`
# Afterwards, run
# `grep "vdw energy" out | awk '{print $3}' > energy_ts.dat`
# to extract the TS energies.

# Run TurboGAP with Many-Body Dispersion (MBD) corrections for the different rotations of the dimer by using `input_dimer_mbd` as input:
# `turbogap_vdw predict > out`
# Afterwards, run
# `grep "vdw energy" out | awk '{print $3}' > energy_mbd.dat`
# to extract the MBD energies.

# Now we plot the potential energy surface (PES) for the dispersion energy calculated with TS and MBD. Run the plotting script in the terminal:
# `python plot_dimer_pes.py`
# Note that we adjust the zero degree energy to the same value for easier comparison of the plots. We want to see the different shapes of PES.

# Next, we run some molecular dynamics (MD) with dispersion corrections. First, generate a cell containing some P4 molecules by running:
# `python generate_p4_cell.py`
# in terminal.

In [23]:
# Visualize the cell
atoms = read("p4_cell.xyz")
viewer = WeasWidget()
viewer.avr.model_style = 1 # ball & stick mode
viewer.from_ase(atoms)
viewer.avr.show_bonded_atoms = True # show bonds across periodic boundaries
viewer

WeasWidget(children=(<weas_widget.base_widget.BaseWidget object at 0x7f17e88f8c10>,))

# Next, we will heat up the structure from its rather unphysical initial state to 800 K by running MD on TurboGAP with dispersion corrections. You should use the file `input_md_init` as the input file. Since we basically just want to create the initial structure at high temperature (but without breaking the molecules), it does not matter which dispersion correction you use for this part. Run the code in the in the terminal using multiple cores as:
`mpirun -np 4 turbogap_vdw md`

# We can visualize the trajectory:

In [24]:
# Visualize the trajectory
atoms = read("trajectory_out.xyz",index=":")
viewer = WeasWidget()
viewer.avr.model_style = 1 # ball & stick mode
viewer.from_ase(atoms)
viewer.avr.show_bonded_atoms = True # show bonds across periodic boundaries
viewer

FileNotFoundError: [Errno 2] No such file or directory: 'trajectory_out.xyz'

# Take the last snapshot of the trajectory as the starting point for quenching:
`tail -110 trajectory_out.xyz > p4_md_quench.xyz`

# Next, we quench the structure from 800 K to 200 K by running TurboGAP with `input_md_quench` as the input. Here you can decide if you want to use TS or cTS by choosing `vdw_type = ts` or `vdw_type = ts+mbd`, respectively. The equilibration part has to be run for quite some time to see clustering and TS is faster, but you might see clustering at higher temperatures (550 K) with cTS where TS alone does not show clustering. If you want, you can change the final temperature by modifying `t_end` to the desired value at this part. Note that you have to modify `t_beg` and `t_end` to this value as well during the equilibration step (next). Run the calculation again in the terminal by:
`mpirun -np 4 turbogap_vdw md`

In [25]:
# Visualize the trajectory again
atoms = read("trajectory_out.xyz",index=":")
viewer = WeasWidget()
viewer.avr.model_style = 1 # ball & stick mode
viewer.from_ase(atoms)
viewer.avr.show_bonded_atoms = True # show bonds across periodic boundaries
viewer

FileNotFoundError: [Errno 2] No such file or directory: 'trajectory_out.xyz'

# Take the last snapshot of the trajectory as the starting point for equilibration:
`tail -110 trajectory_out.xyz > p4_md_equil.xyz`

# Now, we equilibrate for a longer time (200 ps, the previous MD runs were only 20 ps) at 200 K in hopes to see the molecules clustering due to the dispersion interactions between them. You can use `input_md_equil` as the input. Remember to change the temperatures and the dispersion correction method accordingly if you did so in the previous step. Run the equilibration by:
`mpirun -np 4 turbogap_vdw md`

In [26]:
# Visualize the equilibration trajectory
atoms = read("trajectory_out.xyz",index=":")
viewer = WeasWidget()
viewer.avr.model_style = 1 # ball & stick mode
viewer.from_ase(atoms)
viewer.avr.show_bonded_atoms = True # show bonds across periodic boundaries
viewer

FileNotFoundError: [Errno 2] No such file or directory: 'trajectory_out.xyz'

# If you did not see clustering, you could run the equilibration for a longer time or try reducing the temperature.

# We can quantify the clustering by taking the last snapshot of the equilibration by
`tail -110 trajectory_out.xyz > p4_last.xyz`
# and running
`python plot_rdf.py`
# to get the radial distribution function of the first and last snapshot of the equilibration. Note that the first peak is for the covalent bonds so you might have to zoom in a bit.